# GenHMM1d — worked examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamadouyamar/GenHMM1d/blob/master/examples.ipynb)

Every section below **simulates** a model with known parameters and then
**estimates** it back with GenHMM1d, printing true vs. estimated parameters.
All examples are seeded (`np.random.seed(1000)`) and therefore exactly
reproducible. Sample size N = 5000 throughout.

Model classes covered: iid HMMs (Gaussian, Poisson, Student-t),
zero-inflated HMMs, autoregressive HMMs (models M1-M4 of Nasri,
R\u00e9millard & Thioub, 2024, *J. Stat. Comput. Simul.*), and
regime-switching bivariate copulas.


In [1]:
# If running on Google Colab, first install the package:
#   !pip install git+https://github.com/mamadouyamar/GenHMM1d.git
import os, sys, time, warnings
import numpy as np
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())          # allows running from a local clone

from genhmm1d.hmm import HMM
from genhmm1d.ar_hmm import ARHMM

hmm = HMM()
arhmm = ARHMM()

N = 5000                                  # sample size for every example
BURN = 1000                               # burn-in for the simulators
EPS, NINIT, MAXIT = 1e-6, 5, 500          # EM controls

Q2 = np.array([[0.94, 0.06],
               [0.03, 0.97]])
Q3 = np.array([[0.25000, 0.25000, 0.50000],
               [0.37500, 0.58750, 0.03750],
               [0.37500, 0.01875, 0.60625]])
iQ2 = np.array([[0.9, 0.1], [0.1, 0.9]])      # persistent initial_Q (AR)
iQ3 = np.array([[0.8, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]])


def show(name, theta_true, Q_true, theta_hat, Q_hat, t):
    print(f"== {name}  (N={N}, {t:.0f}s) ==")
    print("theta_true:\n", np.asarray(theta_true))
    print("theta_hat :\n", np.round(np.asarray(theta_hat), 3))
    print("Q_true:\n", np.asarray(Q_true))
    print("Q_hat :\n", np.round(np.asarray(Q_hat), 3))

print("setup done")


setup done


## Gaussian HMM (2 regimes)

**Simulated with:** regime means/sd `[[0.0, 1.0], [1.3490, 1.0]]` (50% overlap between the regime densities), transition matrix `Q2`.


In [2]:
theta_gauss = np.array([[0.0, 1.0], [1.3490, 1.0]])   # [mu, sd] per regime
np.random.seed(1000)
y_gauss, _, _ = hmm.SimHMMGen(Q2, 'norm', theta_gauss, N, burn_in=BURN)
t0 = time.time()
out_gauss = hmm.EstHMMGen(np.asarray(y_gauss).reshape(-1, 1), 2, 'norm',
                          max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("Gaussian 2reg", theta_gauss, Q2,
     out_gauss["theta"], out_gauss["Q"], time.time()-t0)


== Gaussian 2reg  (N=5000, 43s) ==
theta_true:
 [[0.    1.   ]
 [1.349 1.   ]]
theta_hat :
 [[0.003 1.   ]
 [1.356 0.989]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.942 0.058]
 [0.039 0.961]]


## Poisson HMM (2 regimes)

**Simulated with:** regime intensities `lambda = [2.0, 9.0]`.


In [3]:
theta_pois = np.array([[2.0], [9.0]])                 # [lambda] per regime
np.random.seed(1000)
y_pois, _, _ = hmm.SimHMMGen(Q2, 'poisson', theta_pois, N, burn_in=BURN)
t0 = time.time()
out_pois = hmm.EstHMMGen(np.asarray(y_pois).reshape(-1, 1), 2, 'poisson',
                         max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("Poisson 2reg", theta_pois, Q2,
     out_pois["theta"], out_pois["Q"], time.time()-t0)


== Poisson 2reg  (N=5000, 1s) ==
theta_true:
 [[2.]
 [9.]]
theta_hat :
 [[1.94 ]
 [9.083]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.946 0.054]
 [0.034 0.966]]


## Student-t HMM (2 regimes)

**Simulated with:** `[dof, loc, scale]` = `[5, 0, 1]` and `[12, 3, 2]`.


Note the estimated `dof` of the second regime: the true value is 12, and a
Student-t with large dof is nearly indistinguishable from a Gaussian, so the
likelihood is almost flat in that direction and the EM can push `dof` toward
infinity (a Gaussian limit). Location, scale, and the transition matrix are
still recovered accurately.


In [4]:
theta_t = np.array([[5.0, 0.0, 1.0], [12.0, 3.0, 2.0]])   # [dof, loc, scale]
np.random.seed(1000)
y_t, _, _ = hmm.SimHMMGen(Q2, 't', theta_t, N, burn_in=BURN)
t0 = time.time()
out_t = hmm.EstHMMGen(np.asarray(y_t).reshape(-1, 1), 2, 't',
                      max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("Student-t 2reg", theta_t, Q2,
     out_t["theta"], out_t["Q"], time.time()-t0)


== Student-t 2reg  (N=5000, 18s) ==
theta_true:
 [[ 5.  0.  1.]
 [12.  3.  2.]]
theta_hat :
 [[ 5.52000000e+00 -3.10000000e-02  9.78000000e-01]
 [ 1.17728517e+31  2.97900000e+00  2.21100000e+00]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.942 0.058]
 [0.036 0.964]]


## Zero-inflated Gaussian HMM (2 regimes)

Regime 0 is a point mass at zero; estimation uses `ZI=1`. **Simulated with:** Gaussian regime `[mu, sd] = [3.0, 1.0]`.


In [5]:
theta_zig = np.array([[0.0, 0.0], [3.0, 1.0]])        # [mu, sd]; row 0 = zero
np.random.seed(1000)
y_zig, _, _ = hmm.SimZIHMMGen(Q2, 'norm', theta_zig, N, burn_in=BURN)
t0 = time.time()
out_zig = hmm.EstHMMGen(np.asarray(y_zig).reshape(-1, 1), 2, 'norm', ZI=1,
                        max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("ZI-Gaussian 2reg", theta_zig, Q2,
     out_zig["theta"], out_zig["Q"], time.time()-t0)
print(f"empirical zero-mass: {(np.asarray(y_zig).ravel() == 0).mean():.3f}")


== ZI-Gaussian 2reg  (N=5000, 1s) ==
theta_true:
 [[0. 0.]
 [3. 1.]]
theta_hat :
 [[0.    0.   ]
 [2.981 0.999]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.946 0.054]
 [0.034 0.966]]
empirical zero-mass: 0.385


## Zero-inflated Gaussian HMM (3 regimes)

Regime 0 = zeros; regimes 1-2 are Gaussians with 50% overlap.


In [6]:
theta_zig3 = np.array([[0.0, 0.0], [0.0, 1.0], [1.3490, 1.0]])
np.random.seed(1000)
y_zig3, _, _ = hmm.SimZIHMMGen(Q3, 'norm', theta_zig3, N, burn_in=BURN)
t0 = time.time()
out_zig3 = hmm.EstHMMGen(np.asarray(y_zig3).reshape(-1, 1), 3, 'norm', ZI=1,
                         max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("ZI-Gaussian 3reg", theta_zig3, Q3,
     out_zig3["theta"], out_zig3["Q"], time.time()-t0)


== ZI-Gaussian 3reg  (N=5000, 12s) ==
theta_true:
 [[0.    0.   ]
 [0.    1.   ]
 [1.349 1.   ]]
theta_hat :
 [[ 0.     0.   ]
 [-0.024  1.003]
 [ 1.34   0.976]]
Q_true:
 [[0.25    0.25    0.5    ]
 [0.375   0.5875  0.0375 ]
 [0.375   0.01875 0.60625]]
Q_hat :
 [[0.246 0.219 0.535]
 [0.384 0.589 0.027]
 [0.386 0.03  0.585]]


## Zero-inflated Poisson HMM (2 regimes)

**Simulated with:** regime 0 = zeros, regime 1 = Poisson(9).


In [7]:
theta_zip = np.array([[0.0], [9.0]])                  # [lambda]; row 0 = zero
np.random.seed(1000)
y_zip, _, _ = hmm.SimZIHMMGen(Q2, 'poisson', theta_zip, N, burn_in=BURN)
t0 = time.time()
out_zip = hmm.EstHMMGen(np.asarray(y_zip).reshape(-1, 1), 2, 'poisson', ZI=1,
                        max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("ZI-Poisson 2reg", theta_zip, Q2,
     out_zip["theta"], out_zip["Q"], time.time()-t0)


== ZI-Poisson 2reg  (N=5000, 1s) ==
theta_true:
 [[0.]
 [9.]]
theta_hat :
 [[0.   ]
 [8.931]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.946 0.054]
 [0.034 0.966]]


## Zero-inflated Poisson HMM (3 regimes)

**Simulated with:** regime 0 = zeros, regimes 1-2 = Poisson(2), Poisson(9).


In [8]:
theta_zip3 = np.array([[0.0], [2.0], [9.0]])
np.random.seed(1000)
y_zip3, _, _ = hmm.SimZIHMMGen(Q3, 'poisson', theta_zip3, N, burn_in=BURN)
t0 = time.time()
out_zip3 = hmm.EstHMMGen(np.asarray(y_zip3).reshape(-1, 1), 3, 'poisson', ZI=1,
                         max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("ZI-Poisson 3reg", theta_zip3, Q3,
     out_zip3["theta"], out_zip3["Q"], time.time()-t0)


== ZI-Poisson 3reg  (N=5000, 3s) ==
theta_true:
 [[0.]
 [2.]
 [9.]]
theta_hat :
 [[0.   ]
 [2.02 ]
 [8.946]]
Q_true:
 [[0.25    0.25    0.5    ]
 [0.375   0.5875  0.0375 ]
 [0.375   0.01875 0.60625]]
Q_hat :
 [[0.241 0.245 0.514]
 [0.371 0.596 0.034]
 [0.391 0.018 0.591]]


## AR(1)-Gaussian HMM (2 regimes)

Autoregressive Gaussian HMM, model **M1** of Nasri, R\u00e9millard & Thioub (2024). **Simulated with:** `[c, phi, sd]` = `[0.0, 0.5, 1.0]` and `[0.7788, 0.5, 1.0]`.


In [9]:
theta_ar = np.array([[0.0, 0.5, 1.0], [0.7788, 0.5, 1.0]])  # [c, phi, sd]
np.random.seed(1000)
y_ar, _, _ = arhmm.SimARXHMMGen(Q2, theta_ar, N, family='norm', burn_in=BURN)
t0 = time.time()
out_ar = arhmm.EstHMMGen_AR(np.asarray(y_ar).ravel(), 2, family='norm',
                            p_AR=1, percentiles=[50], initial_Q=iQ2.copy(),
                            max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("AR(1)-Gaussian 2reg", theta_ar, Q2,
     out_ar["theta"], out_ar["Q"], time.time()-t0)


== AR(1)-Gaussian 2reg  (N=5000, 18s) ==
theta_true:
 [[0.     0.5    1.    ]
 [0.7788 0.5    1.    ]]
theta_hat :
 [[-0.008  0.482  0.942]
 [ 1.056  0.434  0.95 ]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.893 0.107]
 [0.105 0.895]]


## AR(1)-Poisson HMM (2 regimes)

Log-linear autoregressive Poisson, model **M2**: `mu_t = exp(alpha + phi log(1 + Y_(t-1)))`. **Simulated with:** `[alpha, phi]` = `[0.2, 0.3]` and `[1.5, 0.4]`.


In [10]:
theta_arp = np.array([[0.2, 0.3], [1.5, 0.4]])        # [alpha, phi] per regime
np.random.seed(1000)
y_arp, _, _ = arhmm.SimARPoissonGen(Q2, theta_arp, N, burn_in=BURN)
t0 = time.time()
out_arp = arhmm.EstHMMGen_AR(np.asarray(y_arp).ravel(), 2, family='poisson',
                             p_AR=1, percentiles=[50], initial_Q=iQ2.copy(),
                             max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("AR(1)-Poisson 2reg", theta_arp, Q2,
     out_arp["theta"], out_arp["Q"], time.time()-t0)


== AR(1)-Poisson 2reg  (N=5000, 1s) ==
theta_true:
 [[0.2 0.3]
 [1.5 0.4]]
theta_hat :
 [[0.187 0.334]
 [1.544 0.382]]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.945 0.055]
 [0.034 0.966]]


## AR(1) zero-inflated Gaussian HMM (3 regimes)

Model **M3** (observable zero regime). Slowest example (~1 min at N=5000).


In [11]:
theta_arzig = np.array([[0.0, 0.0, 0.0],
                        [0.0, 0.5, 1.0],
                        [0.7788, 0.5, 1.0]])          # [c, phi, sd]; row 0 = zero
np.random.seed(1000)
y_arzig, _, _ = arhmm.SimARZIGaussGen(Q3, theta_arzig, N, burn_in=BURN)
t0 = time.time()
out_arzig = arhmm.EstHMMGen_AR(np.asarray(y_arzig).ravel(), 3, family='norm',
                               p_AR=1, ZI=1, percentiles=[50],
                               initial_Q=iQ3.copy(),
                               max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("AR(1)-ZI-Gaussian 3reg", theta_arzig, Q3,
     out_arzig["theta"], out_arzig["Q"], time.time()-t0)


== AR(1)-ZI-Gaussian 3reg  (N=5000, 16s) ==
theta_true:
 [[0.     0.     0.    ]
 [0.     0.5    1.    ]
 [0.7788 0.5    1.    ]]
theta_hat :
 [[ 0.     0.     0.   ]
 [-0.073  0.48   0.926]
 [ 1.104  0.411  0.898]]
Q_true:
 [[0.25    0.25    0.5    ]
 [0.375   0.5875  0.0375 ]
 [0.375   0.01875 0.60625]]
Q_hat :
 [[0.246 0.396 0.358]
 [0.377 0.495 0.129]
 [0.392 0.062 0.546]]


## AR(1) zero-inflated Poisson HMM (3 regimes)

Model **M4** (hidden zero regime; the Poisson regimes also emit zeros).


In [12]:
theta_arzip = np.array([[0.0, 0.0], [0.3, 0.3], [1.8, 0.4]])  # [alpha, phi]
np.random.seed(1000)
y_arzip, _, _ = arhmm.SimARZIPoisGen(Q3, theta_arzip, N, burn_in=BURN)
t0 = time.time()
out_arzip = arhmm.EstHMMGen_AR(np.asarray(y_arzip).ravel(), 3,
                               family='poisson', p_AR=1, ZI=1,
                               percentiles=[50], initial_Q=iQ3.copy(),
                               max_iter=MAXIT, ninit=NINIT, eps=EPS)
show("AR(1)-ZI-Poisson 3reg", theta_arzip, Q3,
     out_arzip["theta"], out_arzip["Q"], time.time()-t0)


== AR(1)-ZI-Poisson 3reg  (N=5000, 1s) ==
theta_true:
 [[0.  0. ]
 [0.3 0.3]
 [1.8 0.4]]
theta_hat :
 [[0.    0.   ]
 [0.415 0.228]
 [1.78  0.407]]
Q_true:
 [[0.25    0.25    0.5    ]
 [0.375   0.5875  0.0375 ]
 [0.375   0.01875 0.60625]]
Q_hat :
 [[0.26  0.251 0.49 ]
 [0.36  0.581 0.059]
 [0.396 0.014 0.59 ]]


## Regime-switching bivariate copulas: setup

Simulate with `SimHMMCopula`, estimate with `EstHMMCop` (port of `HMMcopula::EstHMMCop` 1.0.4, cross-validated against R). **Simulated with:** Kendall tau per regime `[0.3, 0.7]`.


In [13]:
from genhmm1d.hmm_copula import HMMCopula
hcop = HMMCopula()
TAU_COP = np.array([0.3, 0.7])            # Kendall tau per regime

def show_cop(name, out, t):
    o = np.argsort(out["tau"])
    print(f"== {name}  (N={N}, {t:.0f}s) ==")
    print("tau_true :", TAU_COP)
    print("tau_hat  :", np.round(out["tau"][o], 3))
    print("Q_true:\n", Q2)
    print("Q_hat :\n", np.round(out["Q"][o][:, o], 3))
    if not np.isnan(out["dof"]):
        print("dof_hat  :", round(out["dof"], 2))

print("copula setup done")


copula setup done


## Gaussian copula HMM (2 regimes)


In [14]:
np.random.seed(1000)
y_cgauss, _, _, _ = hcop.SimHMMCopula(Q2, 'gaussian', TAU_COP, N, burn_in=BURN)
t0 = time.time()
out_cgauss = hcop.EstHMMCop(y_cgauss, 2, 'gaussian')
show_cop("Copula gaussian 2reg", out_cgauss, time.time()-t0)


== Copula gaussian 2reg  (N=5000, 15s) ==
tau_true : [0.3 0.7]
tau_hat  : [0.308 0.702]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.949 0.051]
 [0.034 0.966]]


## Clayton copula HMM (2 regimes)


In [15]:
np.random.seed(1000)
y_cclay, _, _, _ = hcop.SimHMMCopula(Q2, 'clayton', TAU_COP, N, burn_in=BURN)
t0 = time.time()
out_cclay = hcop.EstHMMCop(y_cclay, 2, 'clayton')
show_cop("Copula clayton 2reg", out_cclay, time.time()-t0)


== Copula clayton 2reg  (N=5000, 18s) ==
tau_true : [0.3 0.7]
tau_hat  : [0.291 0.685]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.935 0.065]
 [0.038 0.962]]


## Frank copula HMM (2 regimes)


In [16]:
np.random.seed(1000)
y_cfrank, _, _, _ = hcop.SimHMMCopula(Q2, 'frank', TAU_COP, N, burn_in=BURN)
t0 = time.time()
out_cfrank = hcop.EstHMMCop(y_cfrank, 2, 'frank')
show_cop("Copula frank 2reg", out_cfrank, time.time()-t0)


== Copula frank 2reg  (N=5000, 15s) ==
tau_true : [0.3 0.7]
tau_hat  : [0.299 0.698]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.936 0.064]
 [0.041 0.959]]


## Gumbel copula HMM (2 regimes)


In [17]:
np.random.seed(1000)
y_cgum, _, _, _ = hcop.SimHMMCopula(Q2, 'gumbel', TAU_COP, N, burn_in=BURN)
t0 = time.time()
out_cgum = hcop.EstHMMCop(y_cgum, 2, 'gumbel')
show_cop("Copula gumbel 2reg", out_cgum, time.time()-t0)


== Copula gumbel 2reg  (N=5000, 17s) ==
tau_true : [0.3 0.7]
tau_hat  : [0.318 0.706]
Q_true:
 [[0.94 0.06]
 [0.03 0.97]]
Q_hat :
 [[0.94  0.06 ]
 [0.043 0.957]]


### Note: Student-t copula

The Student-t copula HMM (`hcop.EstHMMCop(y, 2, 't')`) works the same way but
is slow (~10 minutes at N = 5000: `t.ppf` is evaluated inside every EM step),
so it is not executed in this notebook.
